In [3]:
#!/usr/bin/env python3
"""
Sensor‑failure prediction: ensembles & nested CV
================================================

Implements:
1. Champion MLP (fixed hyper‑params from earlier tuning)
2. Bagging of that MLP
3. Best SVM (fixed hyper‑params)
4. Stacking ensemble (Bagged‑MLP + SVM → Logistic‑Reg meta)
5. Nested CV for honest performance estimate
6. Probability‑threshold tuning for MCC
"""

# ---------------------------------------------------------------------
# 0) Imports
# ---------------------------------------------------------------------
import warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)

from imblearn.under_sampling import RandomUnderSampler

from sklearn.compose        import ColumnTransformer
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.model_selection import (
    train_test_split,
    RepeatedStratifiedKFold,
    cross_val_score
)
from sklearn.metrics         import (
    matthews_corrcoef,
    make_scorer,
    classification_report,
)
from sklearn.neural_network  import MLPClassifier
from sklearn.svm             import SVC
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import BaggingClassifier, StackingClassifier
from sklearn.base            import clone
from tqdm                    import tqdm

RND   = 42
SCORER = make_scorer(matthews_corrcoef)

# ---------------------------------------------------------------------
# 1) Load & balance the data
# ---------------------------------------------------------------------
df = pd.read_csv("ai4i2020.csv")

FEATURES = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
X, y = df[FEATURES], df["Machine failure"]

rus = RandomUnderSampler(random_state=RND)
X_bal, y_bal = rus.fit_resample(X, y)

# hold‑out split for quick demo (20 %)
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.2, stratify=y_bal, random_state=RND
)

FEATURE_IDX = list(range(len(FEATURES)))          # [0, 1, 2, 3, 4]
preproc = ColumnTransformer(
    [("scale", StandardScaler(), FEATURE_IDX)]
)

# ---------------------------------------------------------------------
# 2) Champion MLP – params from your earlier best run
# ---------------------------------------------------------------------
champ_mlp = Pipeline([
    ("prep", preproc),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(100, 50),
        activation="relu",
        alpha=1e-3,
        solver="adam",
        learning_rate="constant",
        max_iter=500,
        random_state=RND,
        early_stopping=True,
        n_iter_no_change=10,
        validation_fraction=0.15,
    ))
])

# ---------------------------------------------------------------------
# 3) Bagged MLP (10 replicas of the champion)
# ---------------------------------------------------------------------
bag_mlp = BaggingClassifier(
    estimator=champ_mlp,
    n_estimators=10,
    max_samples=1.0,
    max_features=1.0,
    bootstrap=True,
    n_jobs=-1,
    random_state=RND,
)

# ---------------------------------------------------------------------
# 4) Best SVM from earlier tuning
# ---------------------------------------------------------------------
svm_pipe = Pipeline([
    ("prep", preproc),
    ("clf", SVC(
        kernel="rbf",
        C=100,
        gamma="auto",
        probability=True,    # needed for stacking / threshold tuning
        random_state=RND,
    ))
])

# ---------------------------------------------------------------------
# 5) Stacking ensemble (Bagged‑MLP + SVM → LogisticRegression)
# ---------------------------------------------------------------------
stack = StackingClassifier(
    estimators=[("bag_mlp", bag_mlp), ("svm", svm_pipe)],
    final_estimator=LogisticRegression(solver="liblinear"),
    stack_method="predict_proba",
    n_jobs=-1,
    passthrough=False,
)

# Choose which model(s) you want to report; add/remove as desired
models = {
    "Champion MLP": champ_mlp,
    "Bagged MLP":   bag_mlp,
    "SVM":          svm_pipe,
    "Stack":        stack,
}

# ---------------------------------------------------------------------
# 6) Helper: tune probability threshold on a small validation slice
# ---------------------------------------------------------------------
def tune_threshold(probas_val, y_val):
    """Maximise MCC over threshold ∈ [0,1]."""
    best_thr, best_mcc = 0.5, -1
    for thr in np.linspace(0.05, 0.95, 19):
        preds = (probas_val[:, 1] >= thr).astype(int)
        mcc   = matthews_corrcoef(y_val, preds)
        if mcc > best_mcc:
            best_mcc, best_thr = mcc, thr
    return best_thr, best_mcc

# ---------------------------------------------------------------------
# 7) Fit, threshold‑tune, evaluate each model
# ---------------------------------------------------------------------
print("\n=== Hold‑out evaluation with MCC‑optimised threshold ===")
for name, model in models.items():
    # split training into train/val for threshold search (80/20 split)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=RND
    )

    model_fit = clone(model).fit(X_tr, y_tr)

    # threshold tuning
    val_probas = model_fit.predict_proba(X_val)
    thr, _ = tune_threshold(val_probas, y_val)

    # evaluate on *unseen* test set
    test_probas = model_fit.predict_proba(X_test)
    y_pred = (test_probas[:, 1] >= thr).astype(int)
    mcc = matthews_corrcoef(y_test, y_pred)

    print(f"\n▶ {name} │ threshold={thr:.2f} │ Test MCC = {mcc:.4f}")
    print(classification_report(y_test, y_pred))

# ---------------------------------------------------------------------
# 8) Nested CV: unbiased generalisation estimate for best ensemble
# ---------------------------------------------------------------------
print("\n=== Nested 5×2 CV for Stacking ensemble (outer loop) ===")
outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RND)
cv_scores = []

for train_idx, test_idx in tqdm(list(outer.split(X_bal, y_bal))):
    X_tr, X_te = X_bal.iloc[train_idx], X_bal.iloc[test_idx]
    y_tr, y_te = y_bal.iloc[train_idx], y_bal.iloc[test_idx]

    # inner threshold tuning on 20 % of training fold
    X_tr2, X_val2, y_tr2, y_val2 = train_test_split(
        X_tr, y_tr, test_size=0.2, stratify=y_tr, random_state=RND
    )

    model_fold = clone(stack).fit(X_tr2, y_tr2)

    thr_fold, _ = tune_threshold(model_fold.predict_proba(X_val2), y_val2)
    preds_te    = (model_fold.predict_proba(X_te)[:, 1] >= thr_fold).astype(int)

    cv_scores.append(matthews_corrcoef(y_te, preds_te))

print("Outer‑fold MCCs:", np.round(cv_scores, 4))
print(f"Mean ± SD     : {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")



=== Hold‑out evaluation with MCC‑optimised threshold ===

▶ Champion MLP │ threshold=0.45 │ Test MCC = 0.5973
              precision    recall  f1-score   support

           0       0.90      0.65      0.75        68
           1       0.72      0.93      0.81        68

    accuracy                           0.79       136
   macro avg       0.81      0.79      0.78       136
weighted avg       0.81      0.79      0.78       136


▶ Bagged MLP │ threshold=0.45 │ Test MCC = 0.7213
              precision    recall  f1-score   support

           0       0.88      0.84      0.86        68
           1       0.85      0.88      0.86        68

    accuracy                           0.86       136
   macro avg       0.86      0.86      0.86       136
weighted avg       0.86      0.86      0.86       136


▶ SVM │ threshold=0.45 │ Test MCC = 0.8268
              precision    recall  f1-score   support

           0       0.95      0.87      0.91        68
           1       0.88      0.

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:35<00:00,  3.60s/it]

Outer‑fold MCCs: [0.7941 0.7795 0.9122 0.7785 0.785  0.8239 0.7815 0.8089 0.8145 0.8095]
Mean ± SD     : 0.8088 ± 0.0377


In [4]:
df

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,M24855,M,298.8,308.4,1604,29.5,14,0,0,0,0,0,0
9996,9997,H39410,H,298.9,308.4,1632,31.8,17,0,0,0,0,0,0
9997,9998,M24857,M,299.0,308.6,1645,33.4,22,0,0,0,0,0,0
9998,9999,H39412,H,299.0,308.7,1408,48.5,25,0,0,0,0,0,0
